In [ ]:
"""
Week 3 - Day 1
PPO Agent Introduction
=======================
Implementing Proximal Policy
Optimization for dynamic pricing.

Why PPO over DQN?
→ More stable training
→ Better sample efficiency
→ Policy-based (not value-based)
→ Used in ChatGPT training!

Infotact DS/ML Internship — Project 2
"""
import numpy as np
import torch
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]   # RL-Dynamic-Pricing
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("Project Root:", PROJECT_ROOT)
print("SRC Path:", SRC_PATH)
print("Exists:", SRC_PATH.exists())

from environment.pricing_env import (
    DynamicPricingEnv,
    PRICE_LEVELS
)
from agents.ppo.ppo_agent import PPOAgent
from agents.ppo.ppo_network import (
    ActorCriticNetwork
)
from config import PPO

plt.style.use('seaborn-v0_8')
print("✅ PPO modules loaded!")
print(f"\nPPO Config:")
for k, v in PPO.items():
    print(f"  {k:<20}: {v}")

In [ ]:
# Show network architecture
env = DynamicPricingEnv()
net = ActorCriticNetwork(2, 6, [128, 64])
net.print_architecture()

print("\n=== DQN vs PPO COMPARISON ===\n")
print(f"  {'Feature':<25} {'DQN':<20} {'PPO'}")
print("  " + "─" * 60)
comparisons = [
    ('Type', 'Value-based', 'Policy-based'),
    ('Network Output', 'Q-values', 'Probabilities'),
    ('Buffer', 'Replay (off-policy)', 'Rollout (on-policy)'),
    ('Update', 'Every step', 'Every N steps'),
    ('Stability', 'Good', 'Better'),
    ('Action Space', 'Discrete only', 'Any'),
]
for feat, dqn, ppo in comparisons:
    print(f"  {feat:<25} {dqn:<20} {ppo}")

In [ ]:
print("Testing PPO (200 episodes)...\n")
agent  = PPOAgent(env, PPO)
rewards = agent.train(
    n_episodes=200,
    verbose=True
)
print(f"\n✅ PPO works!")
print(f"   Mean: ${np.mean(rewards[-50:]):.0f}")

In [ ]:
# Show what PPO's policy looks like
print("Visualizing PPO policy...\n")

states = [
    ([50, 30], "Full stock, lots of time"),
    ([50, 5],  "Full stock, urgent!"),
    ([10, 30], "Low stock, lots of time"),
    ([10, 5],  "Low stock, urgent!"),
    ([25, 15], "Half and half"),
]

print("PPO Action Probabilities:\n")
for state_vals, desc in states:
    norm  = [
        state_vals[0]/50,
        state_vals[1]/30
    ]
    tensor = torch.FloatTensor([norm])

    with torch.no_grad():
        probs, value = agent.network(tensor)

    probs_np = probs.numpy()[0]
    best_idx = probs_np.argmax()
    best_price = PRICE_LEVELS[best_idx]

    print(f"  State: {desc}")
    print(f"  Probs: {[f'{p:.3f}' for p in probs_np]}")
    print(f"  Best : ${best_price}")
    print(f"  Value: {value.item():.2f}\n")

In [ ]:
print("╔══════════════════════════════════════════╗")
print("║    WEEK 3 DAY 1 — PPO INTRODUCTION      ║")
print("╠══════════════════════════════════════════╣")
print("║  COMPONENTS BUILT:                       ║")
print("║  ✅ Actor-Critic Network                 ║")
print("║  ✅ Rollout Buffer (with GAE)            ║")
print("║  ✅ PPO Clipping Objective               ║")
print("║  ✅ Complete PPO Agent                   ║")
print("╠══════════════════════════════════════════╣")
print("║  PPO vs DQN:                             ║")
print("║  → Policy-based (not value-based)        ║")
print("║  → On-policy (not off-policy)            ║")
print("║  → More stable training                  ║")
print("║  → Used in ChatGPT! 🤖                  ║")
print("╠══════════════════════════════════════════╣")
print("║  Tomorrow → Full PPO Training! 🚀        ║")
print("╚══════════════════════════════════════════╝")